# Comparative Study of Word Embedding Techniques
## Step 3: Implementation and Research Analysis

**Project Title:** Comparative Study of Sparse and Dense Word Embedding Techniques for Legal Document Classification  
**Author:** Antigravity (AI Assistant)  
**Date:** May 16, 2026

### Objective
This notebook implements and analyzes seven core embedding techniques (Sparse and Dense) on a processed legal document dataset. The goal is to understand the semantic and structural differences between these representations before moving to model training.

### Part 1: Data Loading and Verification

We apply embeddings **after** preprocessing because:
1. **Noise Reduction:** Removing stopwords and punctuation reduces the dimensionality of sparse vectors.
2. **Normalization:** Lemmatization ensures that different forms of the same legal term (e.g., 'appeal', 'appeals', 'appealing') are mapped to the same vector index.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure project root is in path
sys.path.append(os.path.abspath('..'))

from src.utils.config import PROCESSED_DATA_PATH, PROCESSED_CSV

# Load data
df = pd.read_csv(os.path.join('..', PROCESSED_DATA_PATH, PROCESSED_CSV))
df = df.dropna(subset=['processed_text', 'case_category'])

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

### Part 2: One-Hot Encoding

**Theory:** One-Hot Encoding represents each word as a binary vector where only one dimension is '1' (corresponding to the word's index in the vocabulary).

**Sparse vs Dense:** One-Hot is the ultimate sparse representation. Most values are zero, and the dimension equals the vocabulary size.

**Dimensionality Problem:** With large legal corpora, One-Hot results in massive, memory-inefficient matrices that lack any semantic relationship between words (e.g., 'judge' and 'court' are as different as 'judge' and 'apple').

In [ ]:
from src.embeddings.bow_tfidf import generate_one_hot

corpus = df['processed_text'].astype(str).tolist()
X_oh, vec_oh = generate_one_hot(corpus, max_features=1000)

print(f"Vocabulary Size: {len(vec_oh.vocabulary_)}")
print(f"Matrix Shape: {X_oh.shape}")
print(f"Sample Vector (First 10 dimensions of first row):\n{X_oh[0, :10].toarray()}")

### Part 3: Bag of Words (BoW)

**Theory:** BoW counts the frequency of each word in a document. It represents a document as a histogram of word counts.

**Limitations:**
1. **Semantic Blindness:** 'Highly recommended' and 'Not recommended' share high similarity despite opposite meanings.
2. **Order Ignorance:** 'The court rules for the plaintiff' vs 'The plaintiff rules for the court' have identical BoW vectors.

In [ ]:
from src.embeddings.bow_tfidf import generate_bow

X_bow, vec_bow = generate_bow(corpus, max_features=1000)
print(f"BoW Feature Matrix Shape: {X_bow.shape}")

# Top Frequent Terms
counts = X_bow.sum(axis=0).A1
words = vec_bow.get_feature_names_out()
freq_df = pd.DataFrame({'word': words, 'count': counts}).sort_values(by='count', ascending=False)
print("\nTop 10 Legal Terms (BoW):")
print(freq_df.head(10))

### Part 4: TF-IDF

**Theory:** TF-IDF weights words based on their frequency in a document (TF) penalizing words that appear frequently across the entire corpus (IDF).

**Legal NLP Utility:** Common legal terms like 'the' or 'said' are filtered out by high IDF, while specific terms like 'habeas' or 'defalcation' gain high weights, making them excellent for classification.

In [ ]:
from src.embeddings.bow_tfidf import generate_tfidf

X_tfidf, vec_tfidf = generate_tfidf(corpus, max_features=1000)
print(f"TF-IDF Matrix Dimensions: {X_tfidf.shape}")

# Top Weighted Terms
tfidf_sums = X_tfidf.sum(axis=0).A1
tfidf_df = pd.DataFrame({'word': vec_tfidf.get_feature_names_out(), 'tfidf': tfidf_sums}).sort_values(by='tfidf', ascending=False)
print("\nTop 10 Important Terms (TF-IDF):")
print(tfidf_df.head(10))

### Part 5: Word2Vec CBOW

**Theory:** Continuous Bag of Words (CBOW) predicts a target word from its surrounding context words. It uses the mean of context vectors.

**Hyperparameters:**
- `vector_size=100`: Dense representation.
- `window=5`: Context window.
- `sg=0`: CBOW mode.

In [ ]:
from src.embeddings.word2vec import train_word2vec

tokenized_corpus = [doc.split() for doc in corpus]
m_cbow = train_word2vec(tokenized_corpus, sg=0)

print("Semantic Similarity Example (CBOW):")
try:
    print(f"Most similar to 'court': {m_cbow.wv.most_similar('court', topn=3)}")
except KeyError:
    print("Word not in vocabulary.")

### Part 6: Word2Vec Skip-Gram

**Theory:** Skip-Gram predicts the context words given a target word. 

**Comparison:** Skip-Gram generally handles rare words better because it creates multiple training pairs for each target-context word relationship, whereas CBOW averages them.

In [ ]:
m_sg = train_word2vec(tokenized_corpus, sg=1)

print("Semantic Similarity Example (Skip-Gram):")
try:
    print(f"Most similar to 'petition': {m_sg.wv.most_similar('petition', topn=3)}")
except KeyError:
    print("Word not in vocabulary.")

### Part 7: GloVe Embeddings

**Theory:** GloVe captures global co-occurrence statistics. Unlike Word2Vec which uses local context, GloVe factorizes the global word-word co-occurrence matrix.

*Note: In production research, we typically use pre-trained vectors (e.g., GloVe 6B).*

In [ ]:
from src.embeddings.glove import load_glove_embeddings

# Mock demonstration for GloVe (Professional projects load external .txt files)
print("GloVe dictionary loading functionality implemented in src/embeddings/glove.py")

### Part 8: FastText

**Theory:** FastText treats words as a bag of character n-grams. This allows it to learn vectors for subwords (e.g., 'anti', 'gravity', 'antigravity').

**Legal NLP Advantage:** Excellent for handling complex legal terminology and spelling variations.

In [ ]:
from src.embeddings.fasttext import train_fasttext

m_ft = train_fasttext(tokenized_corpus)

print("Subword Learning (FastText):")
try:
    print(f"Similarity for 'litigation': {m_ft.wv.most_similar('litigation', topn=3)}")
except KeyError:
    print("Word not in vocabulary.")

### Part 9: Embedding Analysis & Comparison

| Embedding | Sparse/Dense | Semantic Understanding | Handles Rare Words | Context Awareness |
| :--- | :--- | :--- | :--- | :--- |
| **One-Hot** | Sparse | No | Poor | No |
| **BoW** | Sparse | No | Average | No |
| **TF-IDF** | Sparse | Partial | Good | No |
| **Word2Vec** | Dense | Yes | Average | Partial |
| **FastText** | Dense | Yes | Excellent | Partial |

### Visualizations Summary
Generated plots are available in `outputs/figures/`:
1. `vocab_size_comparison.png`: Dimensionality analysis.
2. `tsne_w2v_sg.png`: Semantic neighborhood visualization.
3. `tfidf_importance.png`: Keyword extraction analysis.